<h1> IMPORT DATA </h1>

In [ ]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

movie_ratings = ratings.merge(
    movies,
    on="movieId"
)


In [6]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [8]:
ratings['userId'].nunique()

610

In [9]:
ratings["rating"].mean()

np.float64(3.501556983616962)

In [10]:
ratings["rating"].value_counts().sort_index()

rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64

In [17]:
movie_ratings["title"].value_counts().head(10)

title
Forrest Gump (1994)                          329
Shawshank Redemption, The (1994)             317
Pulp Fiction (1994)                          307
Silence of the Lambs, The (1991)             279
Matrix, The (1999)                           278
Star Wars: Episode IV - A New Hope (1977)    251
Jurassic Park (1993)                         238
Braveheart (1995)                            237
Terminator 2: Judgment Day (1991)            224
Schindler's List (1993)                      220
Name: count, dtype: int64

In [19]:
def get_top_movies(movies, ratings, min_ratings=50, limit=10):
    movie_ratings = ratings.merge(
        movies,
        on="movieId"
    )
    top_movies = (
        movie_ratings.groupby("title")
        .agg(
            rating_count=("rating", "size"),
            rating_mean=("rating", "mean")
        )
        .query("rating_count >= @min_ratings")
        .sort_values("rating_mean", ascending=False)
        .head(limit)
    )
    return top_movies

get_top_movies(movies, ratings)

,rating_count,rating_mean
title,,
"Shawshank Redemption, The (1994)",317,4.429022
"Godfather, The (1972)",192,4.289062
Fight Club (1999),218,4.272936
Cool Hand Luke (1967),57,4.271930
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964),97,4.268041
Rear Window (1954),84,4.261905
"Godfather: Part II, The (1974)",129,4.259690
"Departed, The (2006)",107,4.252336
Goodfellas (1990),126,4.250000


In [21]:
movies["genres"].str.split("|").explode().value_counts()

genres
Drama                 4361
Comedy                3756
Thriller              1894
Action                1828
Romance               1596
Adventure             1263
Crime                 1199
Sci-Fi                 980
Horror                 978
Fantasy                779
Children               664
Animation              611
Mystery                573
Documentary            440
War                    382
Musical                334
Western                167
IMAX                   158
Film-Noir               87
(no genres listed)      34
Name: count, dtype: int64

In [32]:
user_id = 1

user_ratings = ratings[
    ratings["userId"] == user_id
]

user_ratings.head()

user_movies = ratings.merge(
    movies,
    on="movieId"
)

user_movies[
    ["title", "rating", "genres"]
].head(20)

,title,rating,genres
0,Toy Story (1995),4.0,Adventure|Animation|Children|Comedy|Fantasy
1,Grumpier Old Men (1995),4.0,Comedy|Romance
2,Heat (1995),4.0,Action|Crime|Thriller
3,Seven (a.k.a. Se7en) (1995),5.0,Mystery|Thriller
4,"Usual Suspects, The (1995)",5.0,Crime|Mystery|Thriller
5,From Dusk Till Dawn (1996),3.0,Action|Comedy|Horror|Thriller
6,Bottle Rocket (1996),5.0,Adventure|Comedy|Crime|Romance
7,Braveheart (1995),4.0,Action|Drama|War
8,Rob Roy (1995),5.0,Action|Drama|Romance|War
9,Canadian Bacon (1995),5.0,Comedy|War


In [34]:
user_genres = user_movies.copy()

user_genres["genres"] = (
    user_genres["genres"]
    .str.split("|")
)

user_genres = user_genres.explode("genres")

user_genres[["title", "genres", "rating"]].head(20)

,title,genres,rating
0,Toy Story (1995),Adventure,4.0
0,Toy Story (1995),Animation,4.0
0,Toy Story (1995),Children,4.0
0,Toy Story (1995),Comedy,4.0
0,Toy Story (1995),Fantasy,4.0
1,Grumpier Old Men (1995),Comedy,4.0
1,Grumpier Old Men (1995),Romance,4.0
2,Heat (1995),Action,4.0
2,Heat (1995),Crime,4.0
2,Heat (1995),Thriller,4.0


In [35]:
genre_profile = (
    user_genres
    .groupby("genres")["rating"]
    .mean()
    .sort_values(ascending=False)
)
genre_profile.head()

genres
Film-Noir      3.920115
War            3.808294
Documentary    3.797785
Crime          3.658294
Drama          3.656184
Name: rating, dtype: float64